In [3]:
# =========================
# IMPORTS
# =========================

import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import GroupShuffleSplit

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# =========================
# DEVICE SETUP
# =========================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

torch.backends.cudnn.benchmark = True

# =========================
# LOAD DATA
# =========================

df = pd.read_csv("../perf_metrics.csv")

df = df.drop(columns=["pid", "comm"], errors="ignore")

df["label"] = df["label"].replace({
    "normal_idle": "normal",
    "normal_light": "normal",
    "normal_mixed": "normal"
})

drop_cols = [
    "total_free_bytes","involuntary_switches","kmalloc_count","kfree_count",
    "minor_faults","large_page_allocs","write_count","rwsem_read_contentions",
    "cpu_migrations","avg_rwsem_read_wait_ns","rwsem_write_contentions",
    "avg_rwsem_write_wait_ns","max_rwsem_write_wait_ns","avg_mutex_wait_ns",
    "mutex_contentions","max_mutex_wait_ns","poll_count","epoll_count",
    "avg_epoll_latency_ns","kernel_faults","mmap_count","syscall_error_count"
]

df = df.drop(columns=drop_cols, errors="ignore")

# =========================
# FEATURES / LABEL
# =========================

X = df.drop(columns=["label"])
y = df["label"]

le = LabelEncoder()
y = le.fit_transform(y)

scaler = StandardScaler()

X_scaled = scaler.fit_transform(np.log1p(X.drop(columns=["timestamp_s"])))

X_scaled = pd.DataFrame(
    X_scaled,
    columns=X.drop(columns=["timestamp_s"]).columns
)

X_scaled["timestamp_s"] = df["timestamp_s"].values
X_scaled["label"] = y

# =========================
# SEQUENCES
# =========================

SEQ_LEN = 5

X_seq, y_seq, group_ids = [], [], []

for gid, g in X_scaled.groupby("timestamp_s"):
    g = g.sort_values("timestamp_s")

    Xg = g.drop(columns=["label", "timestamp_s"]).values
    yg = g["label"].values

    for i in range(len(Xg) - SEQ_LEN):
        X_seq.append(Xg[i:i+SEQ_LEN])
        y_seq.append(yg[i+SEQ_LEN])
        group_ids.append(gid)

X_seq = np.array(X_seq, dtype=np.float32)
y_seq = np.array(y_seq)
group_ids = np.array(group_ids)

# =========================
# SPLIT
# =========================

gss = GroupShuffleSplit(test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_seq, y_seq, groups=group_ids))

X_train, X_test = X_seq[train_idx], X_seq[test_idx]
y_train, y_test = y_seq[train_idx], y_seq[test_idx]

# =========================
# DATASET
# =========================

class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(
    SeqDataset(X_train, y_train),
    batch_size=32,
    shuffle=True,
    num_workers=0,   # IMPORTANT FIX
    pin_memory=True
)

test_loader = DataLoader(
    SeqDataset(X_test, y_test),
    batch_size=32,
    shuffle=False,
    num_workers=0,   # IMPORTANT FIX
    pin_memory=True
)

# =========================
# MODEL
# =========================

class TransformerBlock(nn.Module):
    def __init__(self, d_model, heads, ff_dim, dropout=0.1):
        super().__init__()

        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=heads,
            batch_first=True
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, d_model)
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        attn, _ = self.attn(x, x, x)
        x = self.norm1(x + self.dropout(attn))

        ff = self.ff(x)
        x = self.norm2(x + self.dropout(ff))

        return x


class Model(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()

        self.project = nn.Linear(input_dim, 64)

        self.block1 = TransformerBlock(64, 2, 64)
        self.block2 = TransformerBlock(64, 2, 64)

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.head = nn.Sequential(
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.project(x)
        x = self.block1(x)
        x = self.block2(x)

        x = x.transpose(1, 2)
        x = self.pool(x).squeeze(-1)

        return self.head(x)

model = Model(
    input_dim=X_train.shape[2],
    num_classes=len(le.classes_)
).to(device)



# =========================
# TRAIN SETUP
# =========================

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

scaler_amp = torch.amp.GradScaler("cuda")

# =========================
# TRAIN LOOP
# =========================

EPOCHS = 50
PATIENCE = 5

best_loss = float("inf")
wait = 0

for epoch in range(EPOCHS):

    model.train()
    train_loss = 0

    for Xb, yb in train_loader:
        Xb = Xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            out = model(Xb)
            loss = criterion(out, yb)

        scaler_amp.scale(loss).backward()
        scaler_amp.step(optimizer)
        scaler_amp.update()

        train_loss += loss.item()

    # =========================
    # VALIDATION
    # =========================

    model.eval()
    val_loss = 0

    with torch.no_grad():
        for Xb, yb in test_loader:
            Xb = Xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            with torch.amp.autocast(device_type="cuda"):
                outputs = model(Xb)
                loss = criterion(outputs, yb)

            val_loss += loss.item()

    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    if val_loss < best_loss:
        best_loss = val_loss
        wait = 0
        torch.save(model.state_dict(), "best.pt")
    else:
        wait += 1

    if wait >= PATIENCE:
        print("Early stopping triggered")
        break

# =========================
# EVALUATION
# =========================

model.load_state_dict(torch.load("best.pt"))
model.eval()

correct, total = 0, 0

with torch.no_grad():
    for Xb, yb in test_loader:
        Xb = Xb.to(device)
        yb = yb.to(device)

        with torch.cuda.amp.autocast():
            out = model(Xb)

        preds = torch.argmax(out, dim=1)

        correct += (preds == yb).sum().item()
        total += yb.size(0)

print("\nTest Accuracy:", correct / total)

Using device: cuda


C:\Users\asad0\AppData\Local\Temp\ipykernel_20448\3759414526.py:235: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1 | Train Loss: 1798.9379 | Val Loss: 1840.1410
Epoch 2 | Train Loss: 1425.9589 | Val Loss: 1654.7090
Epoch 3 | Train Loss: 1330.5719 | Val Loss: 1961.1173
Epoch 4 | Train Loss: 1257.6855 | Val Loss: 1885.0920
Epoch 5 | Train Loss: 1201.4184 | Val Loss: 2333.7682
Epoch 6 | Train Loss: 1153.3991 | Val Loss: 2580.4161
Epoch 7 | Train Loss: 1110.8366 | Val Loss: 2286.6895
Early stopping triggered


C:\Users\asad0\AppData\Local\Temp\ipykernel_20448\3759414526.py:290: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():



Test Accuracy: 0.3526350964292534


inefficient cpu use below

In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import GroupShuffleSplit

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# =========================
# DATA PREPROCESSING (UNCHANGED)
# =========================

df_original = pd.read_csv('../perf_metrics.csv')

df_model = df_original.copy()
df_model = df_model.drop(columns=['pid', 'comm'], errors='ignore')

df_model['label'] = df_model['label'].replace({
    'normal_idle': 'normal',
    'normal_light': 'normal',
    'normal_mixed': 'normal'
})

drop_cols = [
    'total_free_bytes','involuntary_switches','kmalloc_count','kfree_count',
    'minor_faults','large_page_allocs','write_count','rwsem_read_contentions',
    'cpu_migrations','avg_rwsem_read_wait_ns','rwsem_write_contentions',
    'avg_rwsem_write_wait_ns','max_rwsem_write_wait_ns','avg_mutex_wait_ns',
    'mutex_contentions','max_mutex_wait_ns','poll_count','epoll_count',
    'avg_epoll_latency_ns','kernel_faults','mmap_count','syscall_error_count'
]

df_model = df_model.drop(columns=drop_cols, errors='ignore')

X_all = df_model.drop(columns=['label'])
y_all = df_model['label']

le = LabelEncoder()
y_encoded = le.fit_transform(y_all)

scaler = StandardScaler()
X_scaled_full = scaler.fit_transform(np.log1p(X_all.drop(columns=['timestamp_s'])))

X_scaled_df = pd.DataFrame(
    X_scaled_full,
    columns=X_all.drop(columns=['timestamp_s']).columns
)

X_scaled_df['timestamp_s'] = df_model['timestamp_s'].values
X_scaled_df['label'] = y_encoded

# =========================
# SEQUENCE CREATION
# =========================

SEQ_LEN = 5

X_seq, y_seq, session_ids = [], [], []

for session_id, group in X_scaled_df.groupby('timestamp_s'):

    group = group.sort_values(by='timestamp_s')

    X_g = group.drop(columns=['label', 'timestamp_s']).values
    y_g = group['label'].values

    for i in range(len(X_g) - SEQ_LEN):
        X_seq.append(X_g[i:i+SEQ_LEN])
        y_seq.append(y_g[i+SEQ_LEN])
        session_ids.append(session_id)

X_seq = np.array(X_seq, dtype=np.float32)
y_seq = np.array(y_seq)
session_ids = np.array(session_ids)

# =========================
# TRAIN TEST SPLIT
# =========================

gss = GroupShuffleSplit(test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_seq, y_seq, groups=session_ids))

X_train, X_test = X_seq[train_idx], X_seq[test_idx]
y_train, y_test = y_seq[train_idx], y_seq[test_idx]

# =========================
# TORCH DATASET
# =========================

class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(SeqDataset(X_train, y_train), batch_size=8, shuffle=True)
test_loader = DataLoader(SeqDataset(X_test, y_test), batch_size=8)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# =========================
# TRANSFORMER MODEL (PYTORCH)
# =========================

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, ff_dim, dropout=0.1):
        super().__init__()

        self.attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, d_model)
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        x = self.norm1(x + self.dropout(attn_out))

        ff_out = self.ff(x)
        x = self.norm2(x + self.dropout(ff_out))

        return x


class TransformerClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()

        self.input_proj = nn.Linear(input_dim, 64)

        self.block1 = TransformerBlock(64, 2, 64)
        self.block2 = TransformerBlock(64, 2, 64)

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.fc = nn.Sequential(
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.input_proj(x)
        x = self.block1(x)
        x = self.block2(x)

        x = x.transpose(1, 2)
        x = self.pool(x).squeeze(-1)

        return self.fc(x)


model = TransformerClassifier(
    input_dim=X_train.shape[2],
    num_classes=len(le.classes_)
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# =========================
# TRAIN LOOP
# =========================

EPOCHS = 50
PATIENCE = 5

best_loss = float('inf')
patience_counter = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    val_loss = 0
    model.eval()

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            val_loss += loss.item()

    print(f"Epoch {epoch+1}: Train Loss={total_loss:.4f}, Val Loss={val_loss:.4f}")

    if val_loss < best_loss:
        best_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "best_model.pt")
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print("Early stopping")
        break

# =========================
# EVALUATION
# =========================

model.load_state_dict(torch.load("best_model.pt"))
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        outputs = model(X_batch)
        preds = torch.argmax(outputs, dim=1)

        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

print("\n=== PYTORCH TRANSFORMER ===")
print("Test Accuracy:", correct / total)